In [323]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
inf = pd.read_csv("../data/influencers.csv")


In [324]:
inf.isna().sum()                  # count per column
inf.isna().mean().round(3)        # SHARE missing per column (mean of True/False)
inf.isna().any(axis=1).sum()      # rows with AT LEAST ONE missing value

handle           0
niche            0
tier             0
followers_raw    0
country          2
verified         2
avg_likes        5
rate_raw         3
dtype: int64

handle           0.000
niche            0.000
tier             0.000
followers_raw    0.000
country          0.067
verified         0.067
avg_likes        0.167
rate_raw         0.100
dtype: float64

np.int64(12)

In [325]:
inf = pd.read_csv("../data/influencers.csv", na_values=["-", "unknown"])

In [326]:
inf.dropna()                          # drop rows with ANY missing value
inf.dropna(subset=["avg_likes"])      # drop only if avg_likes is missing

,handle,niche,tier,followers_raw,country,verified,avg_likes,rate_raw
0,@creator01,beauty,gold,"117,674",US,no,5219.0,$1170
1,@creator02,fitness,gold,"54,107",US,NO,1271.0,$540
6,@creator07,fitness,silver,"35,155",US,Yes,1057.0,$350
8,@creator09,fitness,bronze,"3,745",CA,no,142.0,$30
10,@creator11,fitness,silver,"26,213",UK,NO,303.0,$260
12,@creator13,beauty,silver,"20,887",UK,yes,387.0,$200
17,@creator18,food,silver,"20,904",US,yes,950.0,$200
18,@creator19,beauty,silver,"15,363",CA,NO,368.0,$150
20,@creator21,fashion,silver,"17,145",US,no,457.0,$170
23,@creator24,fitness,bronze,"7,755",US,yes,369.0,$70


,handle,niche,tier,followers_raw,country,verified,avg_likes,rate_raw
0,@creator01,beauty,gold,"117,674",US,no,5219.0,$1170
1,@creator02,fitness,gold,"54,107",US,NO,1271.0,$540
3,@creator04,fitness,bronze,NaN,US,yes,135.0,$50
4,@creator05,fashion,bronze,"4,016",UK,NO,136.0,NaN
5,@creator06,fitness,bronze,"4,624",US,NaN,183.0,$40
6,@creator07,fitness,silver,"35,155",US,Yes,1057.0,$350
7,@creator08,food,bronze,"4,582",NaN,yes,172.0,$40
8,@creator09,fitness,bronze,"3,745",CA,no,142.0,$30
10,@creator11,fitness,silver,"26,213",UK,NO,303.0,$260
11,@creator12,beauty,bronze,NaN,CA,Yes,112.0,$60


In [327]:
inf.shape                       # (30, 8)
inf.dropna().shape              # (13, 8)  ← lost more than half!

# Plain dropna() is dangerous. One missing country throws away a row that had perfectly good likes. Prefer subset=, so you only drop rows missing the column you actually need.

(30, 8)

(13, 8)

In [328]:
s.fillna(0)                 # a constant
s.fillna(s.median())        # the column's median
s.fillna("Unknown")         # a label (text columns)

0    10.0
1     0.0
2    30.0
3     0.0
dtype: float64

0    10.0
1    20.0
2    30.0
3    20.0
dtype: float64

0       10.0
1    Unknown
2       30.0
3    Unknown
dtype: object

In [329]:
inf["avg_likes"].fillna(inf.groupby("tier")["avg_likes"].transform("median"))

# fillna(0)	0 likes	✗ They obviously get likes
# overall median	369 (a typical bronze/silver number)	✗ Way too low for platinum
# tier median	5,002 (typical platinum)	✓ Much closer to reality

0     5219.0
1     1271.0
2      422.0
3      135.0
4      136.0
5      183.0
6     1057.0
7      172.0
8      142.0
9     5002.0
10     303.0
11     112.0
12     387.0
13    5002.0
14    1507.0
15    1862.0
16     172.0
17     950.0
18     368.0
19     189.0
20     457.0
21     269.0
22    3607.0
23     369.0
24     422.0
25    5292.0
26     137.0
27    2321.0
28    2321.0
29     286.0
Name: avg_likes, dtype: float64

Missing really means zero	fillna(0)	“shares” blank because nobody shared
Missing means unknown, few rows, column is critical	dropna(subset=[...])	target column for a model
Missing means unknown, numeric, you need the rows	fillna(median), ideally per group	avg_likes by tier
Text / category	fillna("Unknown")	country
You’ll take a mean anyway	leave NaN, since .mean() skips it (2.3)	quick summary st

In [330]:
inf["likes_was_missing"] = inf["avg_likes"].isna()

In [331]:
inf.dtypes

handle                   str
niche                    str
tier                     str
followers_raw            str
country                  str
verified                 str
avg_likes            float64
rate_raw                 str
likes_was_missing       bool
dtype: object

In [332]:
# s.astype(float)     # text "1234" → 1234.0
# s.astype(int)       # whole numbers
# s.astype(str)       # anything → text

In [333]:
# pd.Series(["5", None]).astype(int)       # ✗ ValueError: cannot convert NaN to integer
# pd.Series(["5", None]).astype(float)     # ✓ [5.0, NaN]
# pd.Series(["5", None]).astype("Int64")   # ✓ [5, <NA>]  ← capital I = int that allows missing

In [334]:
# pd.to_numeric(..., errors="coerce")

In [335]:
pd.to_numeric(pd.Series(["12", "-", "7"]), errors="coerce")   # [12.0, NaN, 7.0]

0    12.0
1     NaN
2     7.0
dtype: float64

In [336]:
inf["verified"].str.lower().map({"yes": True, "no": False})

0     False
1     False
2     False
3      True
4     False
5       NaN
6      True
7      True
8     False
9      True
10    False
11     True
12     True
13    False
14      NaN
15    False
16    False
17     True
18    False
19     True
20    False
21     True
22     True
23     True
24     True
25    False
26    False
27    False
28    False
29    False
Name: verified, dtype: object

In [337]:
# .replace({...})	left as it was
# .map({...})	becomes NaN

In [338]:
inf["niche"] = inf["niche"].astype("category")


In [339]:
sorted(["bronze", "silver", "gold", "platinum"])
# ['bronze', 'gold', 'platinum', 'silver']   ← silver after platinum?!

['bronze', 'gold', 'platinum', 'silver']

In [340]:
inf["tier"] = pd.Categorical(inf["tier"],
                             categories=["bronze", "silver", "gold", "platinum"],
                             ordered=True)

In [341]:
(inf["tier"] >= "gold").sum()      # gold + platinum creators

np.int64(10)

In [342]:
inf.groupby("tier", observed=True)["avg_likes"].median()

tier
bronze       172.0
silver       422.0
gold        2321.0
platinum    5002.0
Name: avg_likes, dtype: float64

## 2.11 Cheat sheet: Missing values and types

| I want… | Write |
|---|---|
| missing count / share | `df.isna().sum()` / `df.isna().mean()` |
| rows with any missing | `df.isna().any(axis=1).sum()` |
| hidden missing ("-", "unknown") | `read_csv(..., na_values=["-", "unknown"])` |
| drop rows missing one column | `df.dropna(subset=["col"])` |
| fill with constant / median | `s.fillna(0)` / `s.fillna(s.median())` |
| fill per group | `s.fillna(df.groupby("g")["col"].transform("median"))` |
| flag before filling | `df["col_was_missing"] = df["col"].isna()` |
| convert type | `s.astype(float)` / `.astype(int)` / `.astype(str)` |
| int that allows NaN | `s.astype("Int64")` |
| text → number, bad → NaN | `pd.to_numeric(s, errors="coerce")` |
| yes/no → True/False | `s.str.lower().map({"yes": True, "no": False})` |
| category | `s.astype("category")` |
| ordered category | `pd.Categorical(s, categories=[...], ordered=True)` |

**Rules**
- Check `.unique()` for hidden missing values first
- dropna: always use subset= and check shape
- Fill with median (skewed data), per group if groups differ
- Missing = zero → 0. Missing = unknown → don't pretend it's 0
- int can't hold NaN → float or "Int64"
- .map: unlisted → NaN. .replace: unlisted → unchanged

In [343]:
s = pd.Series([10, None, 30, None])
t = pd.Series(["yes", "No", "maybe"])

In [344]:
s.isna().mean()
# 0.5
s.fillna(0).mean()
# 10
s.mean() 
# 20
s.fillna(s.mean()).tolist()
# [10.0, 20.0, 30.0, 20.0]
s.dropna().shape
# (2,)
# pd.Series(["5", None]).astype(int)
#ValueError
pd.to_numeric(pd.Series(["12", "-", "7"]), errors="coerce").sum()
# 19
t.str.lower().map({"yes": True, "no": False}).tolist()
# True, False, NaN
t.str.lower().replace({"yes": True, "no": False}).tolist()
# True, False, maybe
sorted(["bronze", "silver", "gold", "platinum"])
# ['bronze', 'gold', 'platinum', 'silver']

np.float64(0.5)

np.float64(10.0)

np.float64(20.0)

[10.0, 20.0, 30.0, 20.0]

(2,)

np.float64(19.0)

[True, False, nan]

[True, False, 'maybe']

['bronze', 'gold', 'platinum', 'silver']

In [345]:
inf = pd.read_csv("../data/influencers.csv")
# B1. First look

# (a) Show inf.dtypes.
inf.dtypes
# (b) Missing count per column.
inf.isna().sum()
# (c) .unique() of followers_raw (just look at the first few) and of country.
inf['followers_raw'].unique()
inf['country'].unique()
# (d) Comment: which hidden missing values did isna() miss?
#-, unknown

handle               str
niche                str
tier                 str
followers_raw        str
country              str
verified             str
avg_likes        float64
rate_raw             str
dtype: object

handle           0
niche            0
tier             0
followers_raw    0
country          2
verified         2
avg_likes        5
rate_raw         3
dtype: int64

<ArrowStringArray>
['117,674',  '54,107',  '27,584',       '-',   '4,016',   '4,624',  '35,155',
   '4,582',   '3,745', '389,553',  '26,213',  '20,887', '275,935', '125,989',
  '99,636',   '5,531',  '20,904',  '15,363',  '17,145',   '7,224',  '87,109',
   '7,755',  '24,985', '138,279',   '5,780',  '75,612', '120,727',   '6,371']
Length: 28, dtype: str

<ArrowStringArray>
['US', 'UK', nan, 'CA', 'unknown']
Length: 5, dtype: str

In [346]:
# B2. Re-read with na_values

# (a) Re-read the file with na_values=["-", "unknown"]. Store it as inf.
inf = pd.read_csv("../data/influencers.csv", na_values=['-', 'unknown'])
# (b) Missing count per column again.
inf.isna().sum()
# (c) Comment: which columns changed, and by how much?
#followers_raw became 3 from 0, country became 5 from 2
# (d) Share missing per column, rounded to 3.
inf.isna().mean().round(3)
# (e) How many rows have at least one missing value?
inf.isna().any(axis=1).sum() 

handle           0
niche            0
tier             0
followers_raw    3
country          5
verified         2
avg_likes        5
rate_raw         3
dtype: int64

handle           0.000
niche            0.000
tier             0.000
followers_raw    0.100
country          0.167
verified         0.067
avg_likes        0.167
rate_raw         0.100
dtype: float64

np.int64(17)

In [347]:
# B3. The dropna danger

# (a) Shape after plain dropna().
inf.dropna().shape
# (b) Shape after dropna(subset=["avg_likes"]).
inf.dropna(subset=['avg_likes']).shape
# (c) Comment: how many rows did plain dropna() throw away, and why is that bad?
#out of 30 rows, 17 were dropped, which loses more than half the data. 

(13, 8)

(25, 8)

In [348]:
# B4. Fix followers

# (a) Make followers = followers_raw with commas removed, then converted to a number.
inf['followers']= inf['followers_raw'].str.replace(',', "").astype(float)
# (b) Comment: why must it be float, not int? Prove it by trying int in a separate cell and noting the error.
#int cannot accept nans
# (c) Try "Int64" instead. What does the dtype show?
inf['followers']= inf['followers_raw'].str.replace(',', "").astype("Int64")
inf['followers'].dtype
#Int64Dtype()

Int64Dtype()

In [349]:
# B5. Fix rate

# (a) Make rate = rate_raw with $ removed, then converted to float.
inf['rate']= inf['rate_raw'].str.replace('$', "").astype(float)
# (b) How many rates are missing?
inf['rate'].isna().sum()
# (c) Median rate.
inf['rate'].median()

np.int64(3)

np.float64(200.0)

In [350]:
# B6. Fix verified

# (a) Show verified.unique().
inf['verified'].unique()
# (b) Make verified True/False using .str.lower() and .map.
inf['verified_bool']= inf['verified'].str.lower().map({"yes": True, "no": False})
# (c) value_counts(dropna=False) to see True / False / NaN.
inf['verified_bool'].value_counts(dropna=False)

<ArrowStringArray>
['no', 'NO', 'yes', nan, 'Yes']
Length: 5, dtype: str

verified_bool
False    16
True     12
NaN       2
Name: count, dtype: int64

In [351]:
# B7. Three ways to fill avg_likes

# (a) Flag first: make likes_was_missing = avg_likes.isna(). How many?
inf['likes_was_missing']= inf['avg_likes'].isna()
inf['likes_was_missing'].sum()
# (b) fill_zero = avg_likes filled with 0.
inf['fill_zero']= inf['avg_likes'].fillna(0)
# (c) fill_median = avg_likes filled with the overall median.
inf['fill_median']= inf['avg_likes'].fillna(inf['avg_likes'].median())
# (d) fill_tier = avg_likes filled with the tier median (transform, 2.6).
inf['fill_tier']= inf['avg_likes'].fillna(inf.groupby("tier")["avg_likes"].transform("median"))
# (e) Show handle, tier, fill_zero, fill_median and fill_tier for only the rows that were missing (use the flag as a mask).
inf.loc[inf['avg_likes'].isna(), ['handle', 'tier','fill_zero', 'fill_median' , 'fill_tier']]
# (f) Comment: look at @creator10 (platinum). Which method is most believable, and why?
inf[inf['handle']=='@creator10']
#fill_tier, because, being of platinum tier, the number of followers he has, it is very unlikely for him to have 0 or 369 likes.
# (g) Mean of each of the three columns, rounded to 1. Comment: which one drags the mean down most?
inf[['avg_likes','fill_zero', 'fill_median' , 'fill_tier']].mean().round(1)
#fill zero drags it down most. 

np.int64(5)

,handle,tier,fill_zero,fill_median,fill_tier
2,@creator03,silver,0.0,369.0,422.0
9,@creator10,platinum,0.0,369.0,5002.0
16,@creator17,bronze,0.0,369.0,172.0
24,@creator25,silver,0.0,369.0,422.0
27,@creator28,gold,0.0,369.0,2321.0


,handle,niche,tier,followers_raw,country,verified,avg_likes,rate_raw,followers,rate,verified_bool,likes_was_missing,fill_zero,fill_median,fill_tier
9,@creator10,food,platinum,"389,553",UK,yes,NaN,$3890,389553,3890.0,True,True,0.0,369.0,5002.0


avg_likes      1269.3
fill_zero      1057.8
fill_median    1119.3
fill_tier      1335.7
dtype: float64

In [352]:
# B8. Fill country

# (a) Fill missing country with "Unknown".
inf['country']=inf['country'].fillna('Unknown')
# (b) value_counts().
inf['country'].value_counts()

country
US         13
UK          8
Unknown     5
CA          4
Name: count, dtype: int64

In [353]:
# B9. Ordered tier

# (a) sorted(inf["tier"].unique()). Comment: is the order right?
sorted(inf["tier"].unique())
# (b) Make tier an ordered category: bronze < silver < gold < platinum.
inf['tier'] =pd.Categorical(inf["tier"],
                            categories=["bronze", "silver", "gold", "platinum"],
                            ordered=True)
# (c) Count creators with tier >= "gold".
(inf['tier']>='gold').sum()
# (d) Median rate per tier with observed=True. Comment: does the order now make sense when you read it?
inf.groupby("tier", observed=True)["rate"].median()
#yes, order is defined by engagement level

['bronze', 'gold', 'platinum', 'silver']

np.int64(10)

tier
bronze        50.0
silver       220.0
gold        1080.0
platinum    3890.0
Name: rate, dtype: float64

In [354]:
# inf["followers_raw"].str.replace(",", "").astype(int)
inf["followers_raw"].str.replace(",", "").astype(float)
#ValueError: cannot convert float NaN to integer

0     117674.0
1      54107.0
2      27584.0
3          NaN
4       4016.0
5       4624.0
6      35155.0
7       4582.0
8       3745.0
9     389553.0
10     26213.0
11         NaN
12     20887.0
13    275935.0
14    125989.0
15     99636.0
16      5531.0
17     20904.0
18     15363.0
19         NaN
20     17145.0
21      7224.0
22     87109.0
23      7755.0
24     24985.0
25    138279.0
26      5780.0
27     75612.0
28    120727.0
29      6371.0
Name: followers_raw, dtype: float64

In [355]:
inf['avg_likes']= inf["avg_likes"].fillna(inf["avg_likes"].median())
#not saved

In [356]:
# inf = pd.read_csv("../data/influencers.csv")
inf["verified"].str.lower().map({"yes": True, "no": False})
#mapping makes all nonmatching values in map dictionary as NaN

0     False
1     False
2     False
3      True
4     False
5       NaN
6      True
7      True
8     False
9      True
10    False
11     True
12     True
13    False
14      NaN
15    False
16    False
17     True
18    False
19     True
20    False
21     True
22     True
23     True
24     True
25    False
26    False
27    False
28    False
29    False
Name: verified, dtype: object

In [357]:
# inf['rate_was_missing']= inf['rate'].isna()
# inf['rate_was_missing']= inf['rate'].fillna(inf.groupby("niche")["rate"].transform("median"))